# Building Multi LLM Model - Evaluator - Optimizer Model

In [13]:
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display
from pypdf import PdfReader
import gradio as gr
from pathlib import Path
from pydantic import BaseModel
import os

project_root = Path.cwd().parent

In [4]:
# Read the Profile PDF
filePath = project_root / "Resources" / "Rishabh_Maheshwari_Resume (2) (1) (1).pdf"
pdfReader = PdfReader(filePath)
prof_summary = ""
for page in pdfReader.pages:
    text = page.extract_text()
    if text:
        prof_summary += text


# Read Summary file
summ_filePath = project_root / "Resources" / "Personal_Summary.txt"
with open(summ_filePath, "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
load_dotenv(override=True)

import os

openai_api_key=os.getenv('API_TOKEN') 

openai_python_client = OpenAI(api_key=openai_api_key)

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:14]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")

gemini_api_key=os.getenv('GEMINI_API_KEY')
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_client = OpenAI(api_key=gemini_api_key, base_url=gemini_base_url)

if gemini_api_key:
    print(f"OpenAI API Key exists and begins {gemini_api_key[:14]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")

OpenAI API Key exists and begins sk-proj-iocY9p
OpenAI API Key exists and begins AQ.Ab8RN6IK2-H


In [ ]:
# System Prompt
name = "Rishabh Maheshwari"

evaluator_system_prompt = (
    f"You are an evaluator responsible for assessing the quality of an AI Agent's response to a User inquiry.\n\n"
    f"You are given a conversation between a User and an Agent. Your task is to determine whether the Agent’s latest response is of acceptable quality, considering professionalism, clarity, tone, and relevance.\n\n"
    f"The Agent is acting on behalf of {name} and appears on {name}’s website, interacting with visitors who may be potential clients, employers, or professional connections. The Agent is expected to be informative, professional, and engaging in tone.\n\n"
    f"The Agent has been given context about {name}, including their professional summary and LinkedIn profile. Please use this context to inform your evaluation.\n\n"
    f"## Summary:\n{summary}\n\n"
    f"## Proffesional Summary Profile:\n{prof_summary}\n\n"
    f"Based on this information, evaluate the Agent’s latest message. Respond with:\n"
    f"1. **Acceptable** or **Unacceptable**\n"
    f"2. A brief explanation justifying your decision"
)

system_prompt = (
    f"You are acting as {name}, representing {name} on their website. "
    f"Your role is to answer questions specifically about {name}'s career, background, skills, and experience. "
    f"You must faithfully and accurately portray {name} in all interactions. "
    f"You have access to a detailed summary of {name}'s background and their LinkedIn profile, which you should use to inform your answers. "
    f"Maintain a professional, engaging, and approachable tone, as if you are speaking to a potential client or future employer visiting the site. "
    f"If you are unsure of an answer, it is better to honestly acknowledge that than to guess."

    f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{prof_summary}\n\n"
    f"Using this context, please converse naturally and consistently, always staying in character as {name}."
)

In [8]:
class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [9]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = (
        "You are evaluating the most recent response from an AI Agent in the context of a conversation.\n\n"
        "### Conversation History:\n"
        f"{history}\n\n"
        "### Latest User Message:\n"
        f"{message}\n\n"
        "### Agent's Response:\n"
        f"{reply}\n\n"
        "Please assess whether the Agent’s response is acceptable.\n"
    )
    return user_prompt

In [10]:
def evaluate(message, reply, history) -> Evaluation:
    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(message, reply, history)}]
    response = gemini_client.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [15]:
# Submit a User Prompt to LLM1 - DeepSeek
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a Degree?"}]
response = openai_python_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        stream=False
    )

reply = response.choices[0].message.content

In [16]:
reply


'Yes, I have a Bachelor of Technology (B.Tech) degree in Computer Science Engineering from SRM University, which I completed in 2023. My coursework included subjects like Data Structures & Algorithms, Object-Oriented Programming, RDBMS, and Software Engineering. This educational background has equipped me with a solid foundation in technology, which I am actively building upon through practical experience and continuous learning.'